In [5]:
import os
import csv
import re
import pandas as pd


In [6]:
# Folder containing NIH CSVs
in_folder = "NIH/New"

files = [
    "chitosan.csv",
    "hydrogel.csv",
    "PCL.csv",
    "PEG.csv",
    "PLA.csv",
    "PLGA.csv",
    "poly(lactic-co-glycolic acid).csv",
    "polycaprolactone.csv",
    "polyethylene_glycol.csv",
    "polylactic_acid.csv",
    "polymeric_micelle.csv",
]

# Normalize file-stem polymer labels to final names
polymer_mapping = {
    "polycaprolactone": "PCL",
    "pcl": "PCL",
    "polylactic_acid": "PLA",
    "pla": "PLA",
    "polyethylene_glycol": "PEG",
    "peg": "PEG",
    "poly_lactic_co_glycolic_acid": "PLGA",
    "plga": "PLGA",
}

In [7]:
cleaned_dfs = []

for fname in files:
    fpath = os.path.join(in_folder, fname)
    if not os.path.exists(fpath):
        print(f"Missing file: {fpath}")
        continue

    with open(fpath, "r", encoding="utf-8", errors="ignore") as f:
        rows = list(csv.reader(f))

    if len(rows) <= 7:
        print(f"Skipping {fname}: file too short to read header at row 8.")
        continue

    df = pd.DataFrame(rows[8:], columns=rows[7])

    # Standardize column names once
    df.columns = [
        re.sub(r"_+", "_", str(c).strip().lower().replace(" ", "_").replace("/", "_")).strip("_")
        for c in df.columns
    ]

    # Polymer from filename stem
    polymer = os.path.splitext(fname)[0].strip().lower()
    polymer = re.sub(r"[()]", " ", polymer)
    polymer = re.sub(r"[^a-z0-9]+", "_", polymer)
    polymer = re.sub(r"_+", "_", polymer).strip("_")
    polymer = polymer_mapping.get(polymer, polymer)

    df["polymer"] = polymer

    # Deduplicate within file
    if "application_id" in df.columns:
        df = df.drop_duplicates(subset=["application_id"])

    cleaned_dfs.append(df)
    print(f"{fname} -> {df.shape[0]} rows, {df.shape[1]} cols")



chitosan.csv -> 103 rows, 55 cols
hydrogel.csv -> 245 rows, 55 cols
PCL.csv -> 20 rows, 55 cols
PEG.csv -> 152 rows, 55 cols
PLA.csv -> 23 rows, 55 cols
PLGA.csv -> 88 rows, 55 cols
poly(lactic-co-glycolic acid).csv -> 37 rows, 55 cols
polycaprolactone.csv -> 58 rows, 55 cols
polyethylene_glycol.csv -> 56 rows, 55 cols
polylactic_acid.csv -> 30 rows, 55 cols
polymeric_micelle.csv -> 2 rows, 55 cols


In [8]:
# Merge
if not cleaned_dfs:
    raise ValueError("No files loaded. Check in_folder path and filenames.")

nih_merged = pd.concat(cleaned_dfs, ignore_index=True)

# Deduplicate globally
if "application_id" in nih_merged.columns:
    before = nih_merged.shape[0]
    nih_merged = nih_merged.drop_duplicates(subset=["application_id"])
    after = nih_merged.shape[0]
    print(f"Deduplicated NIH dataset: {before} -> {after}")
else:
    print("application_id column not found; skipping global dedup.")

# Quick check
if "polymer" in nih_merged.columns:
    print("Unique polymers:", sorted(nih_merged["polymer"].dropna().unique()))

# Save
out_path = os.path.join(in_folder, "nih_polymer_drug_delivery_cleaned.csv")
nih_merged.to_csv(out_path, index=False)
print(f"Saved: {out_path}")


Deduplicated NIH dataset: 814 -> 627
Unique polymers: ['PCL', 'PEG', 'PLA', 'PLGA', 'chitosan', 'hydrogel']
Saved: NIH/New\nih_polymer_drug_delivery_cleaned.csv
